In [ ]:
import re
import html
import json
import pandas as pd

from pathlib import Path
import ftfy
import emoji
import contractions


# SET PATHS
data_dir = Path("../data/LoveIslandUSA_S7/raw/LoveIslandUSA")
json_files = list(data_dir.glob("*.json"))

print(
    f"N JSON FILES: {len(json_files)}"
)


# MAKE TEXT SAFE
def safe_text(text):

    if text is None:
        return ""

    try:
        if pd.isna(text):
            return ""
    except Exception:
        pass

    return str(text)


# NORMALIZE TEXT
def normalize_text(text):

    text = safe_text(text)

    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = contractions.fix(text)

    return text


# COUNT EMOJIS
def count_emojis(text):

    text = safe_text(text)

    return emoji.emoji_count(text)


# CLEAN TEXT LIGHTLY
def clean_light(text):

    text = normalize_text(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# CLEAN TEXT MORE HEAVILY
def clean_heavy(text):

    text = normalize_text(text)

    text = emoji.demojize(
        text
    )

    text = re.sub(
        r":([a-zA-Z0-9_+-]+):",
        r" [emoji_\1] ",
        text
    )

    text = text.lower()

    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z0-9\s'\[\]_]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# FLATTEN NESTED COMMENTS
def flatten_comments(
    comments,
    post_id,
    post_title,
    post_url,
    depth=0,
    rows=None
):

    if rows is None:
        rows = []

    if not isinstance(comments, list):
        return rows

    for comment in comments:

        if not isinstance(comment, dict):
            continue

        body_raw = comment.get("body")

        if body_raw is None:
            continue

        rows.append({
            "post_id": post_id,
            "post_title": post_title,
            "post_url": post_url,
            "comment_id": comment.get("id"),
            "author": comment.get("author"),
            "body_raw": body_raw,
            "body_light": clean_light(body_raw),
            "body_heavy": clean_heavy(body_raw),
            "body_length": len(safe_text(body_raw)),
            "emoji_count": count_emojis(body_raw),
            "score": comment.get("score"),
            "created_utc": comment.get("created_utc"),
            "parent_id": comment.get("parent_id"),
            "submission": comment.get("submission"),
            "depth": depth
        })

        flatten_comments(
            comments=comment.get("replies", []),
            post_id=post_id,
            post_title=post_title,
            post_url=post_url,
            depth=depth + 1,
            rows=rows
        )

    return rows